In [1]:
import os
import pandas as pd
import torchaudio
from torch.utils.data import Dataset
from pathlib import Path
import math
from typing import List, Tuple, Dict, Any
import tqdm
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

import numpy as np

In [2]:
from pathlib import Path

import pandas as pd
import torch
import torchaudio
from torch.utils.data import Dataset

from pathlib import Path

import pandas as pd
import torch
import torchaudio
from torch.utils.data import Dataset

from kaggle.src.datamodule import SpokenNumbersDataset
from kaggle.src.loss import levenshtein_distance, compute_cer


In [3]:
from sklearn.model_selection import train_test_split
import pandas as pd

root_dir = "kaggle/data/"
full_df = pd.read_csv(Path(root_dir) / "train.csv")

train_classes, test_classes = ['spk_E', 'spk_B', 'spk_A', 'spk_C', 'spk_F', ], ['spk_D']
# full_df['spk_id'].value_counts()

# train_df, test_df = train_test_split(
#     full_df,
#     test_size=0.1,
#     random_state=42,
#     shuffle=True,
# )

train_df = full_df[full_df["spk_id"].isin(train_classes)].reset_index(drop=True)
test_df = full_df[full_df["spk_id"].isin(test_classes)].reset_index(drop=True)

In [4]:
# Staff
def ctc_collate_fn(batch):
    mels, targets, metadatas = zip(*batch)

    # [B, 1, 80, 320]
    mels = torch.stack(mels, dim=0).unsqueeze(1)

    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    targets_concat = torch.cat(targets, dim=0)

    return mels, targets_concat, target_lengths, list(metadatas)
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

In [5]:
from torch.utils.data import DataLoader
train_dataset = SpokenNumbersDataset(
    dataframe=train_df,
    root_dir=root_dir,
    target_frames=320,
    train=True,
)

val_dataset = SpokenNumbersDataset(
    dataframe=test_df,
    root_dir=root_dir,
    target_frames=320,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=ctc_collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    collate_fn=ctc_collate_fn,
)

In [25]:
from kaggle.src.model import CRNNMel
model = CRNNMel(
        backbone_name="resnet18",
        pretrained=True,
        proj_channels=64,
        rnn_hidden_size=128,
        rnn_num_layers=2,
        num_classes=11,
        out_time_steps=40,
    )
x = torch.randn(2, 1, 80, 320)
with torch.no_grad():
    y = model(x)

print("output shape:", y.shape)  # [T, B, C]

# total, trainable = count_parameters(model)
# print(f"total params: {total}")
# print(f"trainable params: {trainable}")
# print(f"params (M): {total / 1e6:.3f}")

output shape: torch.Size([40, 2, 11])


In [7]:
from kaggle.src.model_stage3 import CRNNMelStage3
model = CRNNMelStage3(
        backbone_name="resnet18",
        pretrained=True,
        proj_channels=64,
        rnn_hidden_size=128,
        rnn_num_layers=2,
        num_classes=11,
        out_time_steps=40,
    )
x = torch.randn(2, 1, 80, 320)
with torch.no_grad():
    y = model(x)

print("output shape:", y.shape) 
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

output shape: torch.Size([40, 2, 11])
Total params: 1,133,259
Trainable params: 1,133,259


In [26]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0.0

    for mels, targets, target_lengths, _ in loader:
        mels = mels.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        optimizer.zero_grad()

        log_probs = model(mels) 
        T, B, _ = log_probs.shape
        input_lengths = torch.full((B,), T, dtype=torch.long, device=device)

        loss = criterion(log_probs, targets, input_lengths, target_lengths)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [27]:
@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_refs = []

    for mels, targets, target_lengths, metadatas in loader:
        mels = mels.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        log_probs = model(mels)  
        T, B, _ = log_probs.shape
        input_lengths = torch.full((B,), T, dtype=torch.long, device=device)

        loss = criterion(log_probs, targets, input_lengths, target_lengths)
        total_loss += loss.item()

        preds = greedy_decode(log_probs)
        refs = [m["target_text"] for m in metadatas]

        all_preds.extend(preds)
        all_refs.extend(refs)

    avg_loss = total_loss / len(loader)
    exact_match = sum(p == r for p, r in zip(all_preds, all_refs)) / len(all_refs)
    cer = compute_cer(all_preds, all_refs)

    return avg_loss, exact_match, cer, list(zip(all_preds[:10], all_refs[:10]))

In [28]:
# with torch.no_grad():
#     pred = backbone(torch.rand(1, 3, 80, 320))

In [29]:
from pathlib import Path

import torch
import torch.nn as nn
from kaggle.src.loss import greedy_decode

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

CRNNMel(
  (backbone): FeatureListNet(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act1): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (drop_block): Identity()
        (act1): ReLU(inplace=True)
        (aa): Identity()
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act2): ReLU(inplace=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padd

In [30]:
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
epochs =  200

scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=1.0,
    end_factor=1e-4,
    total_iters=len(train_loader) * epochs
)

save_dir = Path("checkpoints")
save_dir.mkdir(parents=True, exist_ok=True)
best_model_path = save_dir / "best_model.pt"

best_val_acc = float("-inf")
best_epoch = -1

for epoch in range(epochs):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, scheduler, criterion, device
    )
    val_loss, val_acc, val_cer, samples = validate_one_epoch(
        model, val_loader, criterion, device
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"exact_match={val_acc:.4f} | "
        f"CER={val_cer:.4f} ({val_cer * 100:.2f}%)"
    )

    for pred, ref in samples[:3]:
        print(f"  pred={pred} | ref={ref}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_acc": best_val_acc,
                "val_loss": val_loss,
                "val_cer": val_cer,
            },
            best_model_path,
        )

        print(
            f"  -> saved new best model: epoch={epoch:02d}, "
            f"val_acc={best_val_acc:.4f}"
        )

print(f"Best epoch: {best_epoch:02d} | best_val_acc={best_val_acc:.4f}")

Epoch 00 | train_loss=2.8124 | val_loss=2.5675 | exact_match=0.0000 | CER=1.0000 (100.00%)
  pred= | ref=694653
  pred= | ref=718934
  pred= | ref=749375
  -> saved new best model: epoch=00, val_acc=0.0000
Epoch 01 | train_loss=2.2751 | val_loss=1.6322 | exact_match=0.0033 | CER=0.6553 (65.53%)
  pred=69 | ref=694653
  pred=789 | ref=718934
  pred=7 | ref=749375
  -> saved new best model: epoch=01, val_acc=0.0033
Epoch 02 | train_loss=1.1145 | val_loss=0.6753 | exact_match=0.2519 | CER=0.1992 (19.92%)
  pred=694653 | ref=694653
  pred=7894 | ref=718934
  pred=74975 | ref=749375
  -> saved new best model: epoch=02, val_acc=0.2519
Epoch 03 | train_loss=0.5935 | val_loss=0.5070 | exact_match=0.3582 | CER=0.1557 (15.57%)
  pred=694653 | ref=694653
  pred=717934 | ref=718934
  pred=749375 | ref=749375
  -> saved new best model: epoch=03, val_acc=0.3582
Epoch 04 | train_loss=0.4584 | val_loss=0.4219 | exact_match=0.4250 | CER=0.1315 (13.15%)
  pred=694653 | ref=694653
  pred=718934 | ref=718

# predict

In [16]:
device = torch.device("cpu")

# 2. Загружаем веса на CPU
checkpoint = torch.load("checkpoints/best_model.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

# 3. Переводим модель на CPU
model.to(device)
model.eval()

# 4. Делаем пример входа НА CPU
example = torch.randn(1, 1, 80, 320)

# 5. Трейсим
with torch.no_grad():
    scripted_model = torch.jit.trace(model, example)

# 6. Сохраняем
scripted_model.save("asr_model_cpu.pt")

In [17]:
test_df = pd.read_csv(Path(root_dir) / "test.csv")

In [18]:
from pathlib import Path

import torch
import torchaudio
from torch.utils.data import Dataset

__all__ = ["SpokenNumbersDataset"]


class TestSpokenNumbersDataset(Dataset):
    def __init__(
        self,
        dataframe,
        root_dir,
        target_sample_rate=16000,
        n_mels=80,
        n_fft=1024,
        hop_length=256,
        win_length=1024,
        target_frames=320,

        ):
        self.root_dir = Path(root_dir)
        self.data = dataframe.reset_index(drop=True).copy()
        self.target_frames=target_frames
        self.target_sample_rate = target_sample_rate


        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=target_sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            win_length=win_length,
            n_mels=n_mels,
            center=True,
            power=2.0,
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB()
        self.resampler_cache = {}

        self.blank_id = 0
        self.char2idx = {str(i): i + 1 for i in range(10)}
        self.idx2char = {i + 1: str(i) for i in range(10)}

    def __len__(self):
        return len(self.data)

    def _to_mono(self, waveform):
        if waveform.size(0) == 1:
            return waveform
        return waveform.mean(dim=0, keepdim=True)

    def _resample_if_needed(self, waveform, sample_rate):
        if sample_rate == self.target_sample_rate:
            return waveform
        if sample_rate not in self.resampler_cache:
            self.resampler_cache[sample_rate] = torchaudio.transforms.Resample(
                orig_freq=sample_rate,
                new_freq=self.target_sample_rate,
            )
        return self.resampler_cache[sample_rate](waveform)

    def _pad_or_trim_mel(self, mel):
        t = mel.size(1)
        if t < self.target_frames:
            mel = torch.nn.functional.pad(mel, (0, self.target_frames - t))
        elif t > self.target_frames:
            mel = mel[:, :self.target_frames]
        return mel

    def _encode_target(self, text):
        return torch.tensor([self.char2idx[ch] for ch in str(text)], dtype=torch.long)


    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        filename = row["filename"]
        waveform, sample_rate = torchaudio.load(self.root_dir / filename)

        waveform = self._to_mono(waveform)
        waveform = self._resample_if_needed(waveform, sample_rate)

        mel = self.mel_transform(waveform)
        mel = self.db_transform(mel + 1e-8)
        mel = mel.squeeze(0)
        mel = self._pad_or_trim_mel(mel)

        return mel, filename
        
def ctc_test_collate_fn(batch):
    mels, filenames = zip(*batch)

    # [B, 1, 80, 320]
    mels = torch.stack(mels, dim=0).unsqueeze(1)

    return mels, list(filenames)

In [19]:
test_dataset = TestSpokenNumbersDataset(
    dataframe=test_df,
    root_dir=root_dir,
    target_frames=320,
)

In [20]:
test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    collate_fn=ctc_test_collate_fn,
)

In [21]:
data = {
    "filename": [],
    "transcription": [],
}
model.eval()
for mels, filenames in test_loader:
    mels = mels.to(device)
    with torch.no_grad():
        log_probs = model(mels)
        preds = greedy_decode(log_probs)
        data["filename"].append(filenames[0])
        data["transcription"].append(preds[0])

In [22]:
df = pd.DataFrame(data)
df.to_csv("sample_submission.csv", index=False)

In [23]:
mels, filenames = next(iter(test_loader))
with torch.no_grad():
    log_probs = model(mels.to(device))

In [24]:
preds = greedy_decode(log_probs)
list(zip(filenames, preds))

[('test/d2440788a9.mp3', '461694')]

In [126]:
filenames

['test/f73c3590b5.wav']

In [20]:
example = torch.randn(1, 1, 80, 320).to(device)

traced_model = torch.jit.trace(model, example)
traced_model.save("asr_model_scripted.pt")